# 05 — Evento semántico con Gemini (Fase 0)

**Objetivo:** Bitácora estilo PRD a partir de clip + metadatos del evento.


## Prerrequisitos

`GEMINI_API_KEY` en `.env`; salidas de **02** y **04**.


## 1. Setup


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

from _common.io import (
    ensure_scripts_on_path,
    env_or_none,
    load_dotenv_repo,
    read_json,
    read_jsonl,
    setup_logging,
    stage_output_dir,
    write_json,
)
from loguru import logger

ensure_scripts_on_path()
load_dotenv_repo()
setup_logging()


## 2. Configuration


In [2]:
OUT_DIR = stage_output_dir("05_semantic")
EVENTS_PATH = stage_output_dir("04_events") / "events.jsonl"
SEGMENTS_MANIFEST = stage_output_dir("02_segments") / "manifest.json"
GEMINI_MODEL = "gemini-1.5-flash"
SKIPPED = False


## 3. Seleccionar evento y contexto


In [3]:
events = read_jsonl(EVENTS_PATH)
if not events:
    raise RuntimeError("Sin eventos. Ejecute notebook 04.")

event = events[-1]
manifest = read_json(SEGMENTS_MANIFEST)
clip = manifest["clips"][0] if manifest.get("clips") else {}
clip_path = clip.get("path", "")
logger.info("Evento: {} type={}", event.get("event_id"), event.get("type"))


21:59:20 | INFO | Evento: 78eff3b7-c23e-4363-a34d-2b1b6324e819 type=warning


## 4. Llamada Gemini


In [4]:
api_key = env_or_none("GEMINI_API_KEY")
result = {"status": "pending", "event_id": event.get("event_id")}

if not api_key:
    SKIPPED = True
    result = {
        "status": "SKIPPED",
        "reason": "GEMINI_API_KEY no definida en .env",
        "event_id": event.get("event_id"),
    }
    logger.warning(result["reason"])
else:
    import google.generativeai as genai

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(GEMINI_MODEL)
    prompt = f"""Eres un asistente de bitácora industrial VisionOps.
Genera un resumen operativo breve y un JSON con campos: title, severity, description, zone.
Evento detectado: {json.dumps(event, ensure_ascii=False)}
Clip referencia: {clip_path}
Responde primero 2-3 oraciones en español (estilo bitácora) y luego un bloque JSON válido."""
    response = model.generate_content(prompt)
    text = response.text or ""
    result = {
        "status": "ok",
        "event_id": event.get("event_id"),
        "narrative": text,
        "title": event.get("message", "Evento de planta"),
        "severity": event.get("severity", "medium"),
        "description": event.get("message", ""),
        "zone": event.get("zone", "unknown"),
        "raw_model_text": text,
    }

out_path = OUT_DIR / f"semantic_{event.get('event_id', 'last')}.json"
write_json(out_path, result)
result


21:59:20 | WARNING | GEMINI_API_KEY no definida en .env


{'status': 'SKIPPED',
 'reason': 'GEMINI_API_KEY no definida en .env',
 'event_id': '78eff3b7-c23e-4363-a34d-2b1b6324e819'}

## 5. Validación


In [5]:
assert result.get("status") in ("ok", "SKIPPED")
print(f"Estado: {result['status']} → {out_path}")


Estado: SKIPPED → /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/outputs/05_semantic/semantic_78eff3b7-c23e-4363-a34d-2b1b6324e819.json


## Siguiente paso

**[06_generate_heatmap.ipynb](06_generate_heatmap.ipynb)** o **[07_telegram_webhook.ipynb](07_telegram_webhook.ipynb)**
